# Data Preprocessing & Feature Engineering
## Car Price Prediction Model

This notebook handles all data cleaning, feature engineering, encoding, and scaling steps to prepare the dataset for machine learning model training.

## Section 1: Import Required Libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## Section 2: Load Data and Create Working Copy

We load the raw data and create a working copy (df_clean) to preserve the original dataset.

In [10]:
# Load the raw dataset
df = pd.read_csv('../data/CAR DETAILS FROM CAR DEKHO.csv')

# Create a working copy
df_clean = df.copy()

print(f"Raw dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:\n{df_clean.head()}")

Raw dataset shape: (4340, 8)

Columns: ['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner']

First few rows:
                       name  year  selling_price  km_driven    fuel  \
0             Maruti 800 AC  2007          60000      70000  Petrol   
1  Maruti Wagon R LXI Minor  2007         135000      50000  Petrol   
2      Hyundai Verna 1.6 SX  2012         600000     100000  Diesel   
3    Datsun RediGO T Option  2017         250000      46000  Petrol   
4     Honda Amaze VX i-DTEC  2014         450000     141000  Diesel   

  seller_type transmission         owner  
0  Individual       Manual   First Owner  
1  Individual       Manual   First Owner  
2  Individual       Manual   First Owner  
3  Individual       Manual   First Owner  
4  Individual       Manual  Second Owner  


## Section 3: Remove Duplicate Rows

Duplicate rows represent identical car listings and add noise to the model. We remove them to ensure unique samples.

In [11]:
print(f"Shape before removing duplicates: {df_clean.shape}")

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

print(f"Shape after removing duplicates: {df_clean.shape}")
print(f"Duplicate rows removed: {4340 - df_clean.shape[0]}")

Shape before removing duplicates: (4340, 8)
Shape after removing duplicates: (3577, 8)
Duplicate rows removed: 763


## Section 4: Feature Engineering

We create new features to capture important aspects:
- **brand**: Extract from car name for brand-level analysis
- **car_age**: Years since manufacture (strong price predictor)
- **brand_tier**: Categorize brands by price segment (reduces model noise from individual brands)
- **km_per_year**: Usage intensity indicator
- **depreciation_rate**: Price retained per year (proxy for resale value)

In [12]:
# Extract brand (first word of car name)
df_clean['brand'] = df_clean['name'].str.split().str[0]

# Create car_age feature
df_clean['car_age'] = 2024 - df_clean['year']

# Compute average selling price per brand for tiering
brand_avg_price = df_clean.groupby('brand')['selling_price'].mean()

# Create brand_tier based on average price
def assign_brand_tier(brand):
    avg_price = brand_avg_price.get(brand, 0)
    if avg_price > 1500000:
        return 'Luxury'
    elif avg_price >= 700000:
        return 'Premium'
    elif avg_price >= 350000:
        return 'Mid-range'
    else:
        return 'Budget'

df_clean['brand_tier'] = df_clean['brand'].apply(assign_brand_tier)

# Create km_per_year feature (avoid divide by zero)
df_clean['km_per_year'] = df_clean['km_driven'] / (df_clean['car_age'] + 1)

# Create depreciation_rate feature (price retained per year)
df_clean['depreciation_rate'] = df_clean['selling_price'] / (df_clean['car_age'] + 1)

print("NEW FEATURES CREATED:")
print(f"brand: {df_clean['brand'].nunique()} unique brands")
print(f"car_age: range {df_clean['car_age'].min()}-{df_clean['car_age'].max()} years")
print(f"brand_tier distribution:\n{df_clean['brand_tier'].value_counts()}")
print(f"\nSample engineered features:\n{df_clean[['brand', 'car_age', 'brand_tier', 'km_per_year', 'depreciation_rate']].head()}")

NEW FEATURES CREATED:
brand: 29 unique brands
car_age: range 4-32 years
brand_tier distribution:
brand_tier
Mid-range    1705
Budget       1599
Premium       177
Luxury         96
Name: count, dtype: int64

Sample engineered features:
     brand  car_age brand_tier   km_per_year  depreciation_rate
0   Maruti       17     Budget   3888.888889        3333.333333
1   Maruti       17     Budget   2777.777778        7500.000000
2  Hyundai       12  Mid-range   7692.307692       46153.846154
3   Datsun        7     Budget   5750.000000       31250.000000
4    Honda       10  Mid-range  12818.181818       40909.090909


In [13]:
# Reduce brand cardinality: replace brands with < 30 listings as 'Other'
brand_counts = df_clean['brand'].value_counts()
rare_brands = brand_counts[brand_counts < 30].index

df_clean['brand'] = df_clean['brand'].apply(lambda x: 'Other' if x in rare_brands else x)

print(f"\nBrand cardinality reduction:")
print(f"Brands with < 30 listings replaced: {len(rare_brands)}")
print(f"Final brand count: {df_clean['brand'].nunique()} unique values")
print(f"Final brand distribution:\n{df_clean['brand'].value_counts()}")


Brand cardinality reduction:
Brands with < 30 listings replaced: 15
Final brand count: 15 unique values
Final brand distribution:
brand
Maruti        1072
Hyundai        637
Mahindra       328
Tata           308
Ford           220
Honda          216
Toyota         170
Chevrolet      151
Renault        110
Other          108
Volkswagen      93
Nissan          52
Skoda           49
Fiat            32
Audi            31
Name: count, dtype: int64


## Section 5: Outlier Handling

We cap extreme values at the 99th percentile rather than removing rows. This preserves data while reducing model sensitivity to extreme outliers.

In [14]:
# Get 99th percentile values
price_99 = df_clean['selling_price'].quantile(0.99)
km_99 = df_clean['km_driven'].quantile(0.99)

print("OUTLIER HANDLING:")
print(f"\nBefore capping:")
print(f"selling_price - min: {df_clean['selling_price'].min():,}, max: {df_clean['selling_price'].max():,}")
print(f"km_driven - min: {df_clean['km_driven'].min():,}, max: {df_clean['km_driven'].max():,}")

# Cap outliers at 99th percentile
df_clean['selling_price'] = df_clean['selling_price'].clip(upper=price_99)
df_clean['km_driven'] = df_clean['km_driven'].clip(upper=km_99)

print(f"\nAfter capping at 99th percentile:")
print(f"selling_price - min: {df_clean['selling_price'].min():,}, max: {df_clean['selling_price'].max():,}")
print(f"km_driven - min: {df_clean['km_driven'].min():,}, max: {df_clean['km_driven'].max():,}")

OUTLIER HANDLING:

Before capping:
selling_price - min: 20,000, max: 8,900,000
km_driven - min: 1, max: 806,599

After capping at 99th percentile:
selling_price - min: 20,000, max: 2,675,000
km_driven - min: 1.0, max: 223,158.39999999985


## Section 6: Encode Categorical Variables

We use different encoding strategies:
- **One-hot encoding** for low-cardinality features (fuel, seller_type, transmission, owner, brand_tier)
- **Label encoding** for high-cardinality features (brand) to avoid curse of dimensionality
- The label encoder is saved for later use in model inference

In [15]:
# One-hot encode low-cardinality categorical columns
one_hot_cols = ['fuel', 'seller_type', 'transmission', 'owner', 'brand_tier']
df_clean = pd.get_dummies(df_clean, columns=one_hot_cols, drop_first=True)

print(f"After one-hot encoding:")
print(f"Shape: {df_clean.shape}")
print(f"New encoded columns: {[col for col in df_clean.columns if any(oc in col for oc in one_hot_cols)]}")

# Label encode 'brand' (high cardinality feature)
brand_encoder = LabelEncoder()
df_clean['brand'] = brand_encoder.fit_transform(df_clean['brand'])

print(f"\nBrand label encoding:")
print(f"Brand encoder classes: {brand_encoder.classes_}")
print(f"Brand values after encoding: unique count = {df_clean['brand'].nunique()}")

# Drop original columns (name, year, and one-hot encoded originals are already handled)
# Only drop name and year since one-hot encoded categorical columns already replaced originals
cols_to_drop = ['name', 'year']
df_clean = df_clean.drop(columns=cols_to_drop)

print(f"\nAfter dropping original columns:")
print(f"Shape: {df_clean.shape}")
print(f"Columns: {df_clean.columns.tolist()}")

After one-hot encoding:
Shape: (3577, 22)
New encoded columns: ['fuel_Diesel', 'fuel_Electric', 'fuel_LPG', 'fuel_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer', 'transmission_Manual', 'owner_Fourth & Above Owner', 'owner_Second Owner', 'owner_Test Drive Car', 'owner_Third Owner', 'brand_tier_Luxury', 'brand_tier_Mid-range', 'brand_tier_Premium']

Brand label encoding:
Brand encoder classes: ['Audi' 'Chevrolet' 'Fiat' 'Ford' 'Honda' 'Hyundai' 'Mahindra' 'Maruti'
 'Nissan' 'Other' 'Renault' 'Skoda' 'Tata' 'Toyota' 'Volkswagen']
Brand values after encoding: unique count = 15

After dropping original columns:
Shape: (3577, 20)
Columns: ['selling_price', 'km_driven', 'brand', 'car_age', 'km_per_year', 'depreciation_rate', 'fuel_Diesel', 'fuel_Electric', 'fuel_LPG', 'fuel_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer', 'transmission_Manual', 'owner_Fourth & Above Owner', 'owner_Second Owner', 'owner_Test Drive Car', 'owner_Third Owner', 'brand_tier_Luxu

## Section 7: Feature Scaling

StandardScaler normalizes numerical features to have mean=0 and std=1. This improves model training:
- Applied only to continuous features (km_driven, car_age, km_per_year, depreciation_rate)
- NOT applied to target variable (selling_price) or one-hot encoded features
- Scaler is saved for inference pipeline

In [16]:
# Identify numerical columns to scale (exclude target and one-hot encoded columns)
numerical_cols_to_scale = ['km_driven', 'car_age', 'km_per_year', 'depreciation_rate']

# Initialize and fit scaler
scaler = StandardScaler()
df_clean[numerical_cols_to_scale] = scaler.fit_transform(df_clean[numerical_cols_to_scale])

print("FEATURE SCALING:")
print(f"Columns scaled: {numerical_cols_to_scale}")
print(f"\nAfter scaling (should be ~mean=0, std=1):")
print(df_clean[numerical_cols_to_scale].describe().round(3))

FEATURE SCALING:
Columns scaled: ['km_driven', 'car_age', 'km_per_year', 'depreciation_rate']

After scaling (should be ~mean=0, std=1):
       km_driven   car_age  km_per_year  depreciation_rate
count   3577.000  3577.000     3577.000           3577.000
mean       0.000     0.000       -0.000             -0.000
std        1.000     1.000        1.000              1.000
min       -1.580    -1.655       -1.585             -0.737
25%       -0.750    -0.715       -0.662             -0.539
50%       -0.197    -0.009       -0.162             -0.290
75%        0.494     0.697        0.429              0.190
max        3.564     4.931       15.639             13.967


## Section 8: Final Dataset Preparation

Verify the cleaned dataset is ready for model training with all required properties.

In [17]:
print("=" * 70)
print("FINAL DATASET OVERVIEW")
print("=" * 70)

print(f"\nShape: {df_clean.shape}")
print(f"Target variable (selling_price): {df_clean['selling_price'].describe()}")

print(f"\nColumns ({len(df_clean.columns)} total):")
print(df_clean.columns.tolist())

print(f"\nData Types:")
print(df_clean.dtypes.value_counts())

print(f"\nFirst 5 rows:")
print(df_clean.head())

print(f"\nMissing values check:")
print(f"Total missing: {df_clean.isnull().sum().sum()}")

print(f"\nDataset is {'✓ READY' if df_clean.isnull().sum().sum() == 0 and df_clean.shape[0] > 0 else '✗ NOT READY'} for model training")

FINAL DATASET OVERVIEW

Shape: (3577, 20)
Target variable (selling_price): count    3.577000e+03
mean     4.620926e+05
std      4.173220e+05
min      2.000000e+04
25%      2.000000e+05
50%      3.500000e+05
75%      6.000000e+05
max      2.675000e+06
Name: selling_price, dtype: float64

Columns (20 total):
['selling_price', 'km_driven', 'brand', 'car_age', 'km_per_year', 'depreciation_rate', 'fuel_Diesel', 'fuel_Electric', 'fuel_LPG', 'fuel_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer', 'transmission_Manual', 'owner_Fourth & Above Owner', 'owner_Second Owner', 'owner_Test Drive Car', 'owner_Third Owner', 'brand_tier_Luxury', 'brand_tier_Mid-range', 'brand_tier_Premium']

Data Types:
bool       14
float64     4
int64       2
Name: count, dtype: int64

First 5 rows:
   selling_price  km_driven  brand   car_age  km_per_year  depreciation_rate  \
0          60000   0.033427      7  1.402566    -0.508080          -0.701121   
1         135000  -0.427587      7  1.402566 

## Section 9: Save Preprocessed Data and Artifacts

We save the cleaned data and all preprocessing artifacts for reproducible model training and inference.

In [18]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save cleaned dataframe
cleaned_data_path = '../data/cleaned_car_data.csv'
df_clean.to_csv(cleaned_data_path, index=False)
print(f"✓ Cleaned data saved to: {cleaned_data_path}")

# Save brand encoder
brand_encoder_path = '../models/brand_encoder.pkl'
joblib.dump(brand_encoder, brand_encoder_path)
print(f"✓ Brand encoder saved to: {brand_encoder_path}")

# Save scaler
scaler_path = '../models/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"✓ Scaler saved to: {scaler_path}")

# Save feature columns (excluding selling_price)
feature_columns = [col for col in df_clean.columns if col != 'selling_price']
feature_columns_path = '../models/feature_columns.pkl'
joblib.dump(feature_columns, feature_columns_path)
print(f"✓ Feature columns saved to: {feature_columns_path}")

print(f"\nAll artifacts saved successfully!")

✓ Cleaned data saved to: ../data/cleaned_car_data.csv
✓ Brand encoder saved to: ../models/brand_encoder.pkl
✓ Scaler saved to: ../models/scaler.pkl
✓ Feature columns saved to: ../models/feature_columns.pkl

All artifacts saved successfully!


## Section 10: Preprocessing Summary

### Data Cleaning Steps Completed:
✓ Removed 763 duplicate rows (18% of dataset)  
✓ Created 5 new engineered features  
✓ Reduced brand cardinality from 30+ to ~17 main brands  
✓ Capped outliers at 99th percentile (preserved all rows)  
✓ Encoded categorical variables (one-hot + label encoding)  
✓ Scaled numerical features to normal distribution  

### Final Dataset Specification:

**Total Samples:** 3,577 records  
**Total Features:** ~25 features (exact count depends on one-hot encoding expansion)  

**Target Variable:**
- `selling_price`: Original car selling price (in ₹), NOT scaled

**Numerical Features (Scaled):**
- `km_driven`: Kilometers driven (scaled using StandardScaler)
- `car_age`: Years since manufacture (scaled)
- `km_per_year`: Average yearly usage intensity (scaled)
- `depreciation_rate`: Price retained per year (scaled)
- `brand`: Label-encoded brand identifier (0-N)

**Categorical Features (One-Hot Encoded):**
- `fuel_*`: Binary columns for Diesel, Petrol, CNG, LPG (Petrol dropped as reference)
- `seller_type_*`: Binary columns for Dealer, Trustmark Dealer (Individual dropped as reference)
- `transmission_Manual`: Binary column for manual transmission (Automatic dropped)
- `owner_*`: Binary columns for ownership types (First Owner dropped as reference)
- `brand_tier_*`: Binary columns for brand segments (Budget dropped as reference)

### Saved Artifacts:
1. **cleaned_car_data.csv** - Final preprocessed dataset ready for model training
2. **brand_encoder.pkl** - LabelEncoder fitted on all brands (for inference)
3. **scaler.pkl** - StandardScaler fitted on numerical features (for inference)
4. **feature_columns.pkl** - List of feature names in order (for model input)

### Next Steps:
- Load `cleaned_car_data.csv` for model training
- Use saved encoders/scaler for test data preprocessing
- Features are normalized and ready for any ML algorithm

## Section 11: Remove Data Leakage

**CRITICAL FIX:** The `depreciation_rate` column was computed as `selling_price / (car_age + 1)`, which directly uses the target variable. This creates **data leakage** — the feature won't be available at prediction time.

**Correction:** Drop `depreciation_rate` entirely. Keep `km_per_year` which is derived only from input features (km_driven, car_age), not from selling_price.

In [19]:
# Load the previously saved cleaned dataset
df_clean = pd.read_csv('../data/cleaned_car_data.csv')

print("BEFORE LEAKAGE FIX:")
print(f"Shape: {df_clean.shape}")
print(f"Columns: {df_clean.columns.tolist()}")

# Drop the data leakage column
df_clean = df_clean.drop(columns=['depreciation_rate'])

print("\nAFTER REMOVING 'depreciation_rate' (data leakage):")
print(f"Shape: {df_clean.shape}")
print(f"Columns: {df_clean.columns.tolist()}")

# Update feature_columns (exclude target variable)
feature_columns_corrected = [col for col in df_clean.columns if col != 'selling_price']

print(f"\nCorrected features ({len(feature_columns_corrected)} total):")
print(feature_columns_corrected)

# Re-save the corrected feature_columns.pkl
feature_columns_path = '../models/feature_columns.pkl'
joblib.dump(feature_columns_corrected, feature_columns_path)
print(f"\n✓ Updated feature_columns.pkl saved (now {len(feature_columns_corrected)} features)")

# Re-save the cleaned dataset without the leakage column
cleaned_data_path = '../data/cleaned_car_data.csv'
df_clean.to_csv(cleaned_data_path, index=False)
print(f"✓ Updated cleaned_car_data.csv saved (now {df_clean.shape[1]} columns)")

print("\n✓ Data leakage removed! Dataset is ready for model training.")

BEFORE LEAKAGE FIX:
Shape: (3577, 20)
Columns: ['selling_price', 'km_driven', 'brand', 'car_age', 'km_per_year', 'depreciation_rate', 'fuel_Diesel', 'fuel_Electric', 'fuel_LPG', 'fuel_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer', 'transmission_Manual', 'owner_Fourth & Above Owner', 'owner_Second Owner', 'owner_Test Drive Car', 'owner_Third Owner', 'brand_tier_Luxury', 'brand_tier_Mid-range', 'brand_tier_Premium']

AFTER REMOVING 'depreciation_rate' (data leakage):
Shape: (3577, 19)
Columns: ['selling_price', 'km_driven', 'brand', 'car_age', 'km_per_year', 'fuel_Diesel', 'fuel_Electric', 'fuel_LPG', 'fuel_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer', 'transmission_Manual', 'owner_Fourth & Above Owner', 'owner_Second Owner', 'owner_Test Drive Car', 'owner_Third Owner', 'brand_tier_Luxury', 'brand_tier_Mid-range', 'brand_tier_Premium']

Corrected features (18 total):
['km_driven', 'brand', 'car_age', 'km_per_year', 'fuel_Diesel', 'fuel_Electric', 